# Day 36: Random Forest Hyperparameter Tuning
**Author:** Sahil-K-Y  
**Phase:** 3 - Tree Models & SVM  
**Date:** Day 036

---

## 1. Introduction to Hyperparameter Tuning

Hyperparameter tuning optimizes configuration options configured before model fitting starts.
- **Parameters vs Hyperparameters**: Parameters are learned internally from raw training data (e.g. tree split thresholds). Hyperparameters are manually defined prior to training (e.g. max tree depth limits).
- **Regularization tradeoff**: Optimizing tree depth and leaf size configurations prevents estimators from overestimating noise, yielding robust generalize boundaries.

## 2. Key Random Forest Hyperparameters

- `n_estimators`: Total number of trees in the forest.
- `max_depth`: Limits tree levels to prevent deep overfitting.
- `min_samples_split`: Minimum samples a node must contain to consider a split.
- `min_samples_leaf`: Minimum samples a leaf node must contain.
- `max_features`: Number of features to sample at splits (`sqrt` yields $\sqrt{d}$ features).
- `criterion`: The function to evaluate split variance/impurity reductions (`gini` vs. `entropy` vs. `log_loss`).

## 3. RandomizedSearchCV Optimization

Unlike grid search which evaluates all possible combinations, `RandomizedSearchCV` samples a fixed number of configurations randomly from predefined distributions. This makes it computationally faster and extremely effective for large parameter search spaces.

## Exercise 1: Tuning Random Forest using RandomizedSearchCV

We will define a parameter search grid and utilize RandomizedSearchCV to find the best configuration.

### Step 1.1: Loading the Dataset

In [9]:
import numpy as np
from sklearn.datasets import make_classification

# Synthetic classification dataset bana rahe hain jismein 1000 samples aur 20 features hain
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, random_state=42)

print(f"Dataset Shape: X = {X.shape}, y = {y.shape}")

Dataset Shape: X = (1000, 20), y = (1000,)


### Step 1.2: Train-Test Split

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

Train Shape: (800, 20) | Test Shape: (200, 20)


### Step 1.3: Fitting a Baseline Model

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Baseline Random Forest model train karenge (default settings ke sath)
base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
base_rf.fit(X_train, y_train)

# 2. Train aur Test dono ki accuracy check karenge
train_acc = accuracy_score(y_train, base_rf.predict(X_train))
test_acc = accuracy_score(y_test, base_rf.predict(X_test))

print(f"Baseline RF -> Train Accuracy: {train_acc:.4f} | Test Accuracy: {test_acc:.4f}")

Baseline RF -> Train Accuracy: 1.0000 | Test Accuracy: 0.8700


### Step 1.4: Defining the Hyperparameter Search Space

In [12]:
# Define the dictionary containing param_distributions for RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['gini', 'entropy']
}

print("Hyperparameter search space ready hai!")

Hyperparameter search space ready hai!


### Step 1.5: Running RandomizedSearchCV

In [14]:
from sklearn.model_selection import RandomizedSearchCV

# Initialize RandomizedSearchCV
rs = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,          # 20 random combinations test karega (taaki jaldi ho jaye)
    cv=5,               # 5-fold cross validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1           # Fast processing ke liye sabhi CPU cores use karega
)

print("RandomizedSearchCV run ho raha hai... (Thoda wait kijiye)")

# Fit RandomizedSearchCV on training data
rs.fit(X_train, y_train)

print("Hyperparameter Tuning Complete!")

RandomizedSearchCV run ho raha hai... (Thoda wait kijiye)
Hyperparameter Tuning Complete!


### Step 1.6: Analyzing Best Parameters and Score

In [15]:
# Print best parameters and cross-validation score
print("--- Best Configuration Found ---")
print("Best Hyperparameters:", rs.best_params_)
print(f"Best Cross-Validation Accuracy: {rs.best_score_:.4f}")

--- Best Configuration Found ---
Best Hyperparameters: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10, 'criterion': 'gini'}
Best Cross-Validation Accuracy: 0.8975


### Step 1.7: Evaluating the Optimized Estimator

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Sabse best model ko nikaal rahe hain
best_rf = rs.best_estimator_

# Test data par predictions nikaalenge
final_preds = best_rf.predict(X_test)

# Report Card Print karenge
print("--- Optimized Random Forest Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, final_preds):.4f}")
print(f"Precision: {precision_score(y_test, final_preds):.4f}")
print(f"Recall:    {recall_score(y_test, final_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, final_preds):.4f}")

--- Optimized Random Forest Metrics ---
Accuracy:  0.8850
Precision: 0.8571
Recall:    0.8864
F1 Score:  0.8715
